# Análise de Margem por Produto

Neste notebook, analisamos a rentabilidade (Gross Margin %) ao nível do Produto (SKU). Dada a vasta quantidade de produtos, focamo-nos nos produtos com as **Melhores** e **Piores** margens médias globais, e de seguida detalhamos a sua evolução ao longo dos anos e por Região.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.0)

# 1. Carregar os dados
df = pd.read_csv('../data/sales_data_clean.csv')

# Calcular Gross Margin % se não existir
if 'Gross_Margin' not in df.columns:
    df['GP'] = df['Sales_Price'] - df['Production_Cost']
    df['Gross_Margin'] = (df['GP'] / df['Sales_Price']) * 100

print(f"Total de vendas: {len(df):,}")
print(f"Número de produtos únicos (SKUs): {df['Product_Name'].nunique():,}")
print(f"Margem Bruta Média Global: {df['Gross_Margin'].mean():.2f}%")


## 1. Margem Global por Produto (Top 15 e Bottom 15)

Vamos identificar quais os produtos que, em média (agregando todos os anos e regiões), apresentam a maior e a menor margem de lucro percentual.

In [ ]:
# Agrupar por Produto para calcular a margem média
prod_margin = df.groupby('Product_Name').agg(
    Avg_Margin=('Gross_Margin', 'mean'),
    Volume=('Sales_Order_ID', 'count')
).reset_index()

# Filtrar apenas produtos com volume mínimo (ex: > 10 vendas) para evitar outliers estatísticos
prod_margin = prod_margin[prod_margin['Volume'] > 10]

# Top 15 (Melhores Margens)
top_15 = prod_margin.nlargest(15, 'Avg_Margin')

# Bottom 15 (Piores Margens)
bottom_15 = prod_margin.nsmallest(15, 'Avg_Margin')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=top_15, x='Avg_Margin', y='Product_Name', ax=axes[0], palette='Greens_r')
axes[0].set_title('Top 15 Produtos - Melhores Margens (%)', fontweight='bold')
axes[0].set_xlabel('Margem Média (%)')
axes[0].set_ylabel('')

sns.barplot(data=bottom_15, x='Avg_Margin', y='Product_Name', ax=axes[1], palette='Reds_r')
axes[1].set_title('Bottom 15 Produtos - Piores Margens (%)', fontweight='bold')
axes[1].set_xlabel('Margem Média (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("TOP 10 Piores Produtos em Rentabilidade (Apenas produtos com > 10 vendas):")
display(bottom_15.head(10).round(2).set_index('Product_Name'))


## 2. Análise por Ano e Região (Foco nos Produtos Críticos)

Vamos pegar nos **15 produtos com as piores margens** (identificados acima) e visualizar como a sua margem se comportou em cada **Ano** e **Região**.

In [ ]:
# Produtos críticos (Bottom 15)
critical_products = bottom_15['Product_Name'].tolist()

# Filtrar os dados apenas para estes produtos
df_critical = df[df['Product_Name'].isin(critical_products)]

# Agrupar por Produto, Região e Ano
critical_agg = df_critical.groupby(['Product_Name', 'Region', 'Year'])['Gross_Margin'].mean().reset_index()

regions = critical_agg['Region'].unique()

for region in regions:
    d = critical_agg[critical_agg['Region'] == region]
    
    if d.empty:
        continue
        
    plt.figure(figsize=(12, 6))
    pivot = d.pivot(index='Product_Name', columns='Year', values='Gross_Margin').fillna(0)
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=20, cbar_kws={'label': 'Margem Média (%)'})
    plt.title(f'Evolução da Margem (Produtos Críticos) - {region}', fontweight='bold')
    plt.ylabel('Produto')
    plt.xlabel('Ano')
    plt.tight_layout()
    plt.show()


## 3. Tabela Completa: Margem por Produto, Região e Ano

A tabela abaixo contém a margem detalhada para **todos** os produtos (ordenada pelos de pior margem no geral). Devido ao volume de dados, mostramos apenas os 50 primeiros registos.

In [ ]:
# Tabela com todos os produtos
all_prod_agg = df.pivot_table(
    index='Product_Name', 
    columns=['Region', 'Year'], 
    values='Gross_Margin', 
    aggfunc='mean'
)

# Adicionar a média global para ajudar na ordenação
all_prod_agg['Média_Global'] = all_prod_agg.mean(axis=1)

# Calcular e adicionar o volume total
volumes = df.groupby('Product_Name')['Sales_Order_ID'].count()
all_prod_agg['Volume_Total'] = volumes

# Ordenar pelas piores margens globais
all_prod_agg = all_prod_agg.sort_values('Média_Global', ascending=True)

display(all_prod_agg.head(50).round(1))


## 4. Dispersão das Margens (Desvio Padrão por Categoria)

Para perceber se a média de margem de uma categoria é constante e representativa, ou se esconde produtos com margens extremas (muito altas compensando muito baixas), vamos analisar o **Desvio Padrão (Standard Deviation)** e visualizar a distribuição (*Boxplot*) das margens em cada Categoria Principal e Subcategoria.

In [ ]:
# 1. Agrupar dados globais por Categoria Principal
cat_dispersion = df.groupby('Main_Category').agg(
    Avg_Margin=('Gross_Margin', 'mean'),
    Std_Margin=('Gross_Margin', 'std'),
    Min_Margin=('Gross_Margin', 'min'),
    Max_Margin=('Gross_Margin', 'max'),
    Volume=('Sales_Order_ID', 'count')
).reset_index()

print("Dispersão e Desvio Padrão das Margens por Categoria Principal:")
display(cat_dispersion.round(2))

# Visualizar a distribuição (Boxplot) para ver as variações e outliers
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Main_Category', y='Gross_Margin', palette='Set2')
plt.title('Distribuição e Dispersão de Gross Margin (%) por Categoria Principal', fontweight='bold')
plt.ylabel('Gross Margin (%)')
plt.xlabel('Categoria Principal')
plt.axhline(df['Gross_Margin'].mean(), color='red', linestyle='--', label=f"Média Global ({df['Gross_Margin'].mean():.1f}%)")
plt.legend()
plt.tight_layout()
plt.show()

# 2. Desvio Padrão ao nível da Subcategoria (Quais oscilam mais?)
subcat_dispersion = df.groupby(['Main_Category', 'Subcategory']).agg(
    Avg_Margin=('Gross_Margin', 'mean'),
    Std_Margin=('Gross_Margin', 'std'),
    Min_Margin=('Gross_Margin', 'min'),
    Max_Margin=('Gross_Margin', 'max'),
    Volume=('Sales_Order_ID', 'count')
).reset_index()

# Filtrar subcategorias com significância estatística
subcat_dispersion = subcat_dispersion[subcat_dispersion['Volume'] > 50]
top_std_subcats = subcat_dispersion.nlargest(10, 'Std_Margin')

print("\nTop 10 Subcategorias com MAIOR instabilidade/dispersão de Margem (Maior Desvio Padrão):")
display(top_std_subcats.round(2).set_index(['Main_Category', 'Subcategory']))

# Visualizar os Boxplots destas 10 subcategorias mais voláteis
plt.figure(figsize=(14, 6))
sns.boxplot(data=df[df['Subcategory'].isin(top_std_subcats['Subcategory'])], 
            x='Subcategory', y='Gross_Margin', palette='magma')
plt.title('Distribuição de Margem nas 10 Subcategorias Mais Instáveis', fontweight='bold')
plt.ylabel('Gross Margin (%)')
plt.xlabel('Subcategoria')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 5. Produtos com Margem Crítica (Gross Margin < 5%)

Vamos identificar os produtos específicos cujas margens de lucro atingiram níveis críticos (abaixo de 5%), analisando o volume de vendas e o impacto por Região e Ano.

In [ ]:
# Filtrar vendas (linhas individuais) com margem inferior a 5%
low_margin_sales = df[df['Gross_Margin'] < 5]

# Agrupar por Produto, Região e Ano para ver onde estas margens críticas ocorrem
critical_products_detail = low_margin_sales.groupby(['Product_Name', 'Region', 'Year']).agg(
    Avg_Margin=('Gross_Margin', 'mean'),
    Min_Margin=('Gross_Margin', 'min'),
    Max_Margin=('Gross_Margin', 'max'),
    Volume=('Sales_Order_ID', 'count')
).reset_index()

# Ordenar pelo maior volume de vendas com margem crítica
critical_products_detail = critical_products_detail.sort_values('Volume', ascending=False)

print(f"Total de vendas individuais (encomendas) com margem < 5%: {len(low_margin_sales):,}")
print("\nDetalhe dos Produtos com Margens Críticas (< 5%) por Região e Ano (Top 20 por Volume):")
display(critical_products_detail.round(2).head(20))

# Agrupar apenas por Produto para ver o volume total crítico de cada um
critical_prod_summary = low_margin_sales.groupby('Product_Name').agg(
    Avg_Margin=('Gross_Margin', 'mean'),
    Volume=('Sales_Order_ID', 'count')
).reset_index().sort_values('Volume', ascending=False)

if not critical_prod_summary.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=critical_prod_summary.head(15), y='Product_Name', x='Volume', palette='Reds_r')
    plt.title('Top 15 Produtos com Maior Volume de Vendas a Margem < 5%', fontweight='bold')
    plt.xlabel('Volume de Vendas (Nº de Encomendas)')
    plt.ylabel('Produto')
    plt.tight_layout()
    plt.show()
else:
    print("Não existem produtos com margem média inferior a 5%.")


## 6. Margem Líquida de Lucro (Net Profit Margin)

Até agora temos analisado a **Margem Bruta (Gross Margin)**, que considera apenas o Custo de Produção. No entanto, para entender a verdadeira rentabilidade da SkiWell Sports, precisamos analisar a **Margem Líquida (Net Profit Margin)**, que deduz também os **Custos de Expedição/Logística (Shipping Cost)** ao Preço de Venda.

Fórmula: `Net_Margin = (Sales_Price - Production_Cost - Shipping_Cost) / Sales_Price * 100`

In [ ]:
# 1. Calcular Margem Líquida (Net Profit Margin)
df['Net_Margin'] = ((df['Sales_Price'] - df['Production_Cost'] - df['Shipping_Cost']) / df['Sales_Price']) * 100

print(f"Margem Bruta Média Global: {df['Gross_Margin'].mean():.2f}%")
print(f"Margem Líquida Média Global: {df['Net_Margin'].mean():.2f}%")

# 2. Agrupar por Categoria Principal e Região
net_margin_cat = df.groupby(['Main_Category', 'Region']).agg(
    Avg_Gross_Margin=('Gross_Margin', 'mean'),
    Avg_Net_Margin=('Net_Margin', 'mean'),
    Volume=('Sales_Order_ID', 'count')
).reset_index()

# Calcular a "Corrosão Logística" (quantos pontos percentuais a logística destrói)
net_margin_cat['Logistics_Impact_pp'] = net_margin_cat['Avg_Gross_Margin'] - net_margin_cat['Avg_Net_Margin']

print("\nImpacto da Logística na Rentabilidade por Categoria e Região:")
display(net_margin_cat.round(2).sort_values('Logistics_Impact_pp', ascending=False))

# 3. Visualizar Comparação: Margem Bruta vs Margem Líquida por Região
plt.figure(figsize=(12, 6))
net_region = df.groupby('Region')[['Gross_Margin', 'Net_Margin']].mean().reset_index()

# Preparar dados para barplot agrupado (Melt)
net_region_melt = net_region.melt(id_vars='Region', value_vars=['Gross_Margin', 'Net_Margin'], 
                                  var_name='Tipo_Margem', value_name='Margem_pct')

sns.barplot(data=net_region_melt, x='Region', y='Margem_pct', hue='Tipo_Margem', palette='muted')
plt.title('Comparação: Margem Bruta vs Margem Líquida por Região', fontweight='bold', fontsize=14)
plt.ylabel('Margem Média (%)')
plt.xlabel('Região')
plt.legend(title='Tipo de Margem', labels=['Margem Bruta (Gross)', 'Margem Líquida (Net)'])
plt.tight_layout()
plt.show()

# 4. Top 15 Produtos com as Piores Margens Líquidas Globais
net_prod = df.groupby('Product_Name').agg(
    Avg_Net_Margin=('Net_Margin', 'mean'),
    Volume=('Sales_Order_ID', 'count')
).reset_index()

net_prod = net_prod[net_prod['Volume'] > 10].nsmallest(15, 'Avg_Net_Margin')

plt.figure(figsize=(12, 6))
sns.barplot(data=net_prod, y='Product_Name', x='Avg_Net_Margin', palette='Reds_r')
plt.title('Top 15 Produtos com as Piores Margens Líquidas Globais', fontweight='bold', fontsize=14)
plt.xlabel('Margem Líquida Média (%)')
plt.ylabel('Produto')
plt.tight_layout()
plt.show()


## 7. Análise Interativa: Produtos com Margem Líquida Crítica (< N%)

Nesta secção, utilizamos um controlo interativo para definir um limite (Threshold) de Margem Líquida (%). O objetivo é identificar instantaneamente quantos produtos, qual o volume de vendas (encomendas) e qual a receita (Revenue) total gerada com uma rentabilidade líquida abaixo do valor que definir.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.ticker as mticker

# Assegurar que a Net_Margin está calculada
if 'Net_Margin' not in df.columns:
    df['Net_Margin'] = ((df['Sales_Price'] - df['Production_Cost'] - df['Shipping_Cost']) / df['Sales_Price']) * 100

out_critical_net = widgets.Output()

w_threshold = widgets.FloatSlider(
    value=10.0,
    min=-20.0,
    max=50.0,
    step=1.0,
    description='Threshold Net Margin (%):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

def plot_critical_net_margin(change=None):
    with out_critical_net:
        clear_output(wait=True)
        n = w_threshold.value
        
        # Filtrar vendas (linhas) com Margem Líquida abaixo do threshold (N)
        df_crit = df[df['Net_Margin'] < n]
        
        if df_crit.empty:
            print(f"Não existem vendas com Margem Líquida inferior a {n}%.")
            return
            
        print(f"=== Resumo de Vendas com Margem Líquida < {n}% ===")
        print(f"Total de Encomendas (Volume): {len(df_crit):,}")
        print(f"Revenue Total em Risco: €{df_crit['Sales_Price'].sum():,.2f}")
        print(f"Número de Produtos Únicos afetados: {df_crit['Product_Name'].nunique():,}\n")
        
        # Agrupar por produto
        prod_summary = df_crit.groupby('Product_Name').agg(
            Volume_Encomendas=('Sales_Order_ID', 'count'),
            Revenue_Total=('Sales_Price', 'sum'),
            Avg_Net_Margin=('Net_Margin', 'mean')
        ).reset_index().sort_values('Volume_Encomendas', ascending=False)
        
        print(f"Tabela: Detalhe por Produto com Margem Líquida < {n}% (Ordenado por Volume de Encomendas)")
        display(prod_summary.round(2).head(20).set_index('Product_Name'))
        
        # Visualização dos produtos que mais receita geram com margens críticas
        prod_summary_rev = prod_summary.sort_values('Revenue_Total', ascending=False)
        
        plt.figure(figsize=(12, 6))
        ax = sns.barplot(data=prod_summary_rev.head(15), y='Product_Name', x='Revenue_Total', palette='Reds_r')
        plt.title(f'Top 15 Produtos com Maior Revenue e Margem Líquida < {n}%', fontweight='bold', fontsize=14)
        plt.xlabel('Revenue Total (€)')
        plt.ylabel('Produto')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"€{x/1000:.0f}k"))
        plt.tight_layout()
        plt.show()

w_threshold.observe(plot_critical_net_margin, names='value')
display(w_threshold, out_critical_net)
plot_critical_net_margin()
